# Demo: neural-network regression in PyTorch, and wide networks as Gaussian processes

TIF285 *Learning from data*, lecture on artificial neural networks (book sections 27.1–27.4).

This notebook follows the blackboard derivations step by step:

1. **Forward pass.** A network is a stack of affine maps and activations. We check that `nn.Linear` computes $\boldsymbol{z}^l = \boldsymbol{y}^{l-1}\boldsymbol{W}^l + \boldsymbol{b}^l$, count the parameters, and look at the signals in a randomly initialized network.
2. **Backpropagation = automatic differentiation.** We compute the gradients with the $\delta^l$ recursion from the backprop section and check that `loss.backward()` gives the same numbers.
3. **Training loop.** Each of its five lines corresponds to one step of the algorithm on the board.
4. **What a trained network does and does not do.** Interpolation works well; extrapolation does not, and the result depends on the activation function and the random seed.
5. **Bonus: Wide random networks are Gaussian processes.** We sample many random networks and look at the distribution of their outputs and at the kernel, for linear, $\tanh$ and ReLU activations.

Notation follows section 27.2. Signals are **row vectors**, and $w^l_{ij}$ connects node $i$ in layer $l-1$ to node $j$ in layer $l$. For a batch of $N_d$ inputs stacked as rows, $\boldsymbol{Z}^l = \boldsymbol{Y}^{l-1}\boldsymbol{W}^l + \boldsymbol{b}^l$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)   # small problems: no reason to use float32
plt.rcParams.update({"figure.dpi": 100, "font.size": 11})

## 1. Forward pass: a network is a stack of `nn.Linear` layers and activations

`nn.Sequential` applies its entries one after another, left to right, each taking the output of the previous one.

`nn.Linear(n_in, n_out)` stores a weight matrix of shape `(n_out, n_in)` and computes `x @ weight.T + bias`. So **`layer.weight.T` is our $\boldsymbol{W}^l$** ($N_{l-1}\times N_l$), and a batch of inputs is processed row by row, exactly as in $\boldsymbol{Z} = \boldsymbol{X}\boldsymbol{W} + \boldsymbol{b}$.

The constructor of a `nn.Linear(n_in, n_out)` call allocates a weight tensor of shape (`n_out`, `n_in`) and a bias of length `n_out`, and fills both with random numbers. PyTorch's default draws every weight and bias from $\mathcal{U}(−1/\sqrt{n_\mathrm{in}}, 1/\sqrt{n_\mathrm{in}})$, where $n_\mathrm{in} = N_{l−1}$ (fan-in).

Below is the 2–3–1 network from the first exercise in section 27.2, with sigmoid activations everywhere. We check the forward pass by hand and count the parameters ($3\times 3 + 4 = 13$).

In the code below, `net231(X)` propagates the signal forward. PyTorch also records each operation and the intermediate signals it will need when applying the chain rule for derivatives. `loss.backward()` then uses that record to compute all the gradients in one backward pass. Here, we just want the signals and use `detach()` to drop the gradient records from torch output.

In [ ]:
torch.manual_seed(0)
net231 = nn.Sequential(nn.Linear(2, 3), nn.Sigmoid(), nn.Linear(3, 1), nn.Sigmoid()) # a sequence of operations

X = torch.randn(4, 2)                      # four instances of input data (part d of that exercise)
W1, b1 = net231[0].weight.T, net231[0].bias   # W1 is 2 x 3
W2, b2 = net231[2].weight.T, net231[2].bias   # W2 is 3 x 1

Y1 = torch.sigmoid(X @ W1 + b1)            # 4 x 3: one row per data instance
Y2 = torch.sigmoid(Y1 @ W2 + b2)           # 4 x 1

print("by hand  :", Y2.detach().ravel().numpy()) # detach drops the gradient records from torch output
print("nn module:", net231(X).detach().ravel().numpy())
print("number of parameters:", sum(p.numel() for p in net231.parameters()))

The network used in the rest of the demo has **one input, three hidden layers of width 10, and one linear output node**. That gives $(1+1)\cdot 10 + 2\cdot(10+1)\cdot 10 + (10+1)\cdot 1 = 251$ parameters.

In [ ]:
def make_net(activation=nn.Tanh, width=10, depth=3):
    layers, n_in = [], 1
    for _ in range(depth):
        layers += [nn.Linear(n_in, width), activation()]
        n_in = width
    layers.append(nn.Linear(n_in, 1))      # linear output node for regression
    return nn.Sequential(*layers)

print(make_net())
print("number of parameters:", sum(p.numel() for p in make_net().parameters()))

### Signals in a randomly initialized network

Before any training, the weights are random numbers. PyTorch's default draws each weight and bias from $\mathcal{U}(-1/\sqrt{N_{l-1}}, 1/\sqrt{N_{l-1}})$. We push a grid of inputs $x \in [-3, 3]$ through the network one layer at a time and store the outputs $y^l_j(x)$ of every node.

- **Left three panels:** the 10 node signals in each hidden layer, for one initialization. Layer 1 contains shifted and stretched $\tanh$ functions of $x$. Each later layer forms new functions from combinations of the previous layer's signals.
- **Right panel:** the network output $y(x)$ for eight different initializations. Each initialization gives a different smooth function. Training will select one of them by adjusting the weights, and section 5 asks what this family of random functions looks like when the width goes to infinity.

Note also that the signals get **smaller from layer to layer**. The default weight variance is $1/(3N_{l-1})$, which is below the value $1/N_{l-1}$ at which a $\tanh$ network passes signals through unchanged in size. In a deep network the signals, and with them the gradients, therefore shrink geometrically with depth. Tuning the initialization to avoid this ("criticality") is discussed in the book's chapter on ANNs in the large-width limit.

In [ ]:
x_demo = torch.linspace(-3, 3, 200).reshape(-1, 1)

def layer_signals(net, x):
    # Forward pass one module at a time; keep the output y^l after every activation.
    signals, y = [], x
    for module in net:
        y = module(y)
        if not isinstance(module, nn.Linear):
            signals.append(y.detach().numpy())
    return signals, y.detach().numpy()

torch.manual_seed(3)
signals, _ = layer_signals(make_net(), x_demo)

fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
for l, y in enumerate(signals):
    ax[l].plot(x_demo, y, lw=1)
    ax[l].set(title=f"hidden layer {l+1}: $y^{l+1}_j(x)$", xlabel="$x$")
for seed in range(8):
    torch.manual_seed(100 + seed)
    _, y_out = layer_signals(make_net(), x_demo)
    ax[3].plot(x_demo, y_out, lw=1.5)
ax[3].set(title="output $y(x)$: 8 random initializations", xlabel="$x$")
plt.tight_layout()

## 2. Backpropagation is reverse-mode automatic differentiation

PyTorch records every operation on a tensor that has `requires_grad=True`. Calling `.backward()` on a scalar runs the chain rule backwards through that record. As a warm-up, check against a derivative we can do by hand: $g(x) = \sin(x^2)\,e^{-x}$ has $g'(x) = \bigl[2x\cos(x^2) - \sin(x^2)\bigr]e^{-x}$.

In [ ]:
x = torch.linspace(0, 3, 7, requires_grad=True)
g = torch.sin(x**2) * torch.exp(-x)
g.sum().backward()          # .backward() needs a scalar; d(sum g)/dx_i = g'(x_i)

exact = (2*x*torch.cos(x**2) - torch.sin(x**2)) * torch.exp(-x)
print("autograd:", x.grad.numpy().round(6))
print("by hand :", exact.detach().numpy().round(6))

### The backprop equations, written out for the 2–3–1 network

For a network, `loss.backward()` computes exactly the backprop recursion. In batch form, with one row per data instance and sigmoid activations ($f' = y(1-y)$), and with cost $C = \tfrac12\sum (y^2 - t)^2$ summed over the batch:
$$
\boldsymbol{\Delta}^2 = (\boldsymbol{Y}^2 - \boldsymbol{T})\odot f'(\boldsymbol{Z}^2), \qquad
\boldsymbol{\Delta}^1 = \bigl(\boldsymbol{\Delta}^2 (\boldsymbol{W}^2)^\intercal\bigr)\odot f'(\boldsymbol{Z}^1),
$$
$$
\frac{\partial C}{\partial \boldsymbol{W}^l} = (\boldsymbol{Y}^{l-1})^\intercal\boldsymbol{\Delta}^l, \qquad
\frac{\partial C}{\partial \boldsymbol{b}^l} = \sum_{\text{rows}} \boldsymbol{\Delta}^l, \qquad \boldsymbol{Y}^0 \equiv \boldsymbol{X}.
$$
We reuse `X`, `Y1`, `Y2`, `W2` from the forward pass in section 1, add some targets, and compare with autograd. PyTorch stores the gradient in the same `(n_out, n_in)` layout as the weights, so we compare our $\partial C/\partial\boldsymbol{W}^l$ with `weight.grad.T`.

In [ ]:
T = torch.tensor([[0.0], [1.0], [1.0], [0.0]])     # targets for the four inputs

# --- by hand: the backprop equations ---------------------------------------------
# no_grad() stops the gradient record from being built in the first place
with torch.no_grad():
    D2 = (Y2 - T) * Y2 * (1 - Y2)               # 4 x 1   output-layer errors
    D1 = (D2 @ W2.T) * Y1 * (1 - Y1)            # 4 x 3   back-propagated errors
    grads_hand = {"W1": X.T @ D1,  "b1": D1.sum(0),     # 2 x 3, 3
                  "W2": Y1.T @ D2, "b2": D2.sum(0)}     # 3 x 1, 1

# --- autograd -----------------------------------------------------------------
net231.zero_grad()
loss = 0.5 * ((net231(X) - T)**2).sum()
loss.backward()
grads_auto = {"W1": net231[0].weight.grad.T, "b1": net231[0].bias.grad,
              "W2": net231[2].weight.grad.T, "b2": net231[2].bias.grad}

for k in grads_hand:
    print(f"{k}: max |hand - autograd| = {(grads_hand[k] - grads_auto[k]).abs().max().item():.1e}")
print("dC/dW1 by hand:\n", grads_hand["W1"].numpy().round(5))

## 3. Data and the training loop

The target function is $\sin^2 x$, with Gaussian noise $\sigma = 0.05$. We use 20 training points and 20 validation points, drawn uniformly in $[0, 2]$. Predictions are made on $[-1, 4]$, so that we also see what happens *outside* the data.

The training loop has five lines. Each one does one step of the learning algorithm:

| line | board |
|---|---|
| `opt.zero_grad()` | gradients are *accumulated* by PyTorch, so reset them |
| `pred = net(x_train)` | forward pass: store all $\boldsymbol{z}^l$, $\boldsymbol{y}^l$ |
| `loss = mse(pred, y_train)` | cost $C$ |
| `loss.backward()` | backward pass: $\boldsymbol{\delta}^L$, then $\boldsymbol{\delta}^{l}$ for $l = L-1,\ldots,1$ |
| `opt.step()` | update all parameters, $\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta\,\nabla_{\boldsymbol{\theta}} C$ (here with Adam instead of plain gradient descent) |

We use the full training set in every step (batch gradient descent), since there are only 20 points. With large data sets, each step would use a mini-batch instead.

In [ ]:
def target(x):
    return np.sin(x)**2

rng = np.random.default_rng(2026)
sigma_noise = 0.05
# training and validation data
x_tr = rng.uniform(0, 2, 20); y_tr = target(x_tr) + sigma_noise*rng.normal(size=20)
x_va = rng.uniform(0, 2, 20); y_va = target(x_va) + sigma_noise*rng.normal(size=20)
# prediction grid
x_grid = np.linspace(-1, 4, 300)

col = lambda a: torch.tensor(a).reshape(-1, 1)    # PyTorch wants (N_data, N_features)
xt, yt, xv, yv, xg = col(x_tr), col(y_tr), col(x_va), col(y_va), col(x_grid)

In [ ]:
def train(net, epochs=1500, lr=0.01):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    mse = nn.MSELoss()
    hist = []
    for epoch in range(epochs):
        opt.zero_grad()
        pred = net(xt)
        loss = mse(pred, yt)
        loss.backward()
        opt.step()
        with torch.no_grad():                       # validation: no gradients, no updates
            hist.append((loss.item(), mse(net(xv), yv).item()))
    return np.array(hist)

torch.manual_seed(1)
net = make_net(nn.Tanh)
hist = train(net)
with torch.no_grad():
    y_net = net(xg).numpy().ravel()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(hist[:, 0], label="training")
ax[0].semilogy(hist[:, 1], label="validation")
ax[0].axhline(sigma_noise**2, color="k", ls=":", label=r"$\sigma^2_{\rm noise}$")
ax[0].set(xlabel="epoch", ylabel="MSE", title="Learning curves"); ax[0].legend()

ax[1].plot(x_grid, target(x_grid), "k--", label=r"$\sin^2 x$")
ax[1].plot(x_tr, y_tr, "o", color="C0", label="training data")
ax[1].plot(x_va, y_va, "s", mfc="none", color="C1", label="validation data")
ax[1].plot(x_grid, y_net, color="C3", lw=2, label="network (tanh)")
ax[1].axvspan(0, 2, color="0.92", zorder=0)
ax[1].set(xlabel="$x$", ylim=(-0.6, 1.6), title="Fit and extrapolation"); ax[1].legend(fontsize=9)
plt.tight_layout()

Things to point out:

- The validation loss levels off near $\sigma^2_{\rm noise}$, as it should for a good fit. The training loss ends up *below* $\sigma^2_{\rm noise}$: with 251 parameters and 20 data points, the network has started to fit the noise. This overfitting is mild here, but it is the reason we monitor a validation set.
- Inside $[0, 2]$ (shaded) the fit is good. Outside, the network has no information about $\sin^2 x$. It returns *something smooth*, determined by the architecture and the random initialization, not by the physics.

## 4. Activation function and random seed

We train five networks with $\tanh$ and five with ReLU, each from a different random initialization.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, (name, act) in enumerate([("tanh", nn.Tanh), ("ReLU", nn.ReLU)]):
    preds = []
    for seed in range(5):
        torch.manual_seed(seed)
        n = make_net(act)
        train(n)
        with torch.no_grad():
            preds.append(n(xg).numpy().ravel())
        ax[a].plot(x_grid, preds[-1], lw=1.5)
    ax[a].plot(x_grid, target(x_grid), "k--")
    ax[a].plot(x_tr, y_tr, "ko", ms=4)
    ax[a].axvspan(0, 2, color="0.92", zorder=0)
    ax[a].set(xlabel="$x$", title=f"{name}: five random seeds", ylim=(-0.6, 1.6))
plt.tight_layout()

- **ReLU** networks are piecewise linear, so they extrapolate linearly. **tanh** networks saturate, so they extrapolate to roughly constant values.
- The seeds agree where there is data and disagree outside it. This spread is a crude measure of our ignorance there. A single trained network reports no uncertainty at all.

This raises a question: before seeing any data, what kind of functions does a network with random weights represent? This is the prior over functions induced by the prior over weights, and in the wide limit we can compute it.

## 5. Wide random networks are Gaussian processes

Consider one hidden layer of width $N$, activation function $f$, and a linear output, and write the network output as $\phi(x)$. Write it as a sum of the contributions $g_j$ of the hidden nodes:
$$
\phi(x) = b^{2} + \sum_{j=1}^{N} g_j(x),
\qquad
g_j(x) = w^{2}_{j}\,f\bigl(x\,w^{1}_{j} + b^{1}_{j}\bigr),
$$
$$
w^1_j \sim \mathcal{N}(0,\sigma_{w1}^2),\;
b^1_j \sim \mathcal{N}(0,\sigma_{b1}^2),\;
w^2_j \sim \mathcal{N}(0,\sigma_w^2/N),\;
b^2 \sim \mathcal{N}(0,\sigma_b^2).
$$
All weights are independent. Each $g_j$ is a random basis function $f(x\,w^1_j + b^1_j)$ with a random amplitude $w^2_j$, and it depends only on node $j$'s own weights.

**Mean and covariance.** $\mathbb{E}[g_j(x)] = 0$ because $w^2_j$ has zero mean, so $\mathbb{E}[\phi(x)] = 0$. In the double sum $\mathbb{E}[\phi(x)\phi(x')] = \sigma_b^2 + \sum_{j,k}\mathbb{E}[g_j(x)\,g_k(x')]$, the terms with $j\neq k$ vanish, because different nodes are independent and have zero mean. The $N$ terms with $j=k$ are identical, each equal to $(\sigma_w^2/N)\,\mathbb{E}\bigl[f(x\,w^1_j+b^1_j)\,f(x'w^1_j+b^1_j)\bigr]$. Hence
$$
\kappa(x,x') \equiv \mathbb{E}[\phi(x)\phi(x')] = \sigma_b^2 + \sigma_w^2\,\mathbb{E}\bigl[f(x\,w^1_j+b^1_j)\,f(x'w^1_j+b^1_j)\bigr],
$$
where the expectation is over $w^1_j, b^1_j$ and is the same for every $j$. $\phi(x)$ and $\phi(x')$ are strongly correlated when a typical hidden node responds similarly at $x$ and $x'$.

**Gaussianity.** The $g_j$ are i.i.d. *across nodes* $j$, while $g_j(x)$ and $g_j(x')$ are correlated *across inputs*, since they come from the same node. By the multivariate central limit theorem applied to the vectors $\bigl(g_j(x_1),\ldots,g_j(x_n)\bigr)$, the vector $\bigl(\phi(x_1),\ldots,\phi(x_n)\bigr)$ becomes jointly Gaussian as $N\to\infty$, for *any* set of inputs. That is the definition of a GP: $\phi \sim \mathcal{GP}\bigl[0,\ \kappa(x,x')\bigr]$.
The $1/N$ in the output-weight variance makes $\kappa$ independent of $N$. PyTorch's default initialization draws weights from $\mathcal{U}(-1/\sqrt{n_{\rm in}}, 1/\sqrt{n_{\rm in}})$, whose variance is $1/(3 n_{\rm in})$: the same $1/n_{\rm in}$ scaling.

In [ ]:
sw1, sb1, sw, sb = 1.5, 1.0, 1.0, 0.1      # standard deviations of the weight priors

activations = {"linear": lambda z: z, "tanh": torch.tanh, "relu": torch.relu}

def random_nets(x, N, act_name, M, seed=0, chunk=500):
    # Outputs phi(x) of M independently initialized one-hidden-layer nets of width N at inputs x.
    # Returns an array of shape (M, len(x)).
    f = activations[act_name]
    g = torch.Generator().manual_seed(seed)
    x = torch.tensor(x).reshape(1, 1, -1)
    out = []
    for m0 in range(0, M, chunk):
        m = min(chunk, M - m0)
        w1 = sw1 * torch.randn(m, N, 1, generator=g)
        b1 = sb1 * torch.randn(m, N, 1, generator=g)
        w2 = sw / np.sqrt(N) * torch.randn(m, N, 1, generator=g)
        b2 = sb * torch.randn(m, 1, generator=g)
        out.append((w2 * f(w1 * x + b1)).sum(dim=1) + b2)
    return torch.cat(out).numpy()

### 5a. The output distribution becomes Gaussian as the width grows

Below are histograms of $\phi(x_0)$ at a single input $x_0 = 1$, over 20 000 random initializations of a ReLU network. For $N=1$ the output is $w^2\max(0,\cdot)$ plus a small bias, so half the networks give $\phi(x_0)\approx 0$ and the distribution is far from Gaussian. The black curve is a Gaussian with the same mean and variance as the samples. By $N \approx 100$ the histogram is indistinguishable from it. The sample variance, shown in each title, stays about the same as $N$ grows: that is the effect of the $1/N$ in the output-weight variance.

In [ ]:
x0 = np.array([1.0])
fig, ax = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
zz = np.linspace(-4.5, 4.5, 200)
for a, N in enumerate([1, 3, 10, 100]):
    phi0 = random_nets(x0, N, "relu", M=20000, seed=N).ravel()
    ax[a].hist(phi0, bins=np.linspace(-4.5, 4.5, 91), density=True, color="C2", alpha=0.6)
    mu, var = phi0.mean(), phi0.var()
    ax[a].plot(zz, np.exp(-(zz - mu)**2/(2*var))/np.sqrt(2*np.pi*var), "k", lw=1.5)
    ax[a].set(title=f"width $N={N}$, variance {var:.2f}", xlabel=r"$\phi(x_0)$")
ax[0].set_ylabel("density")
plt.tight_layout()

### 5b. The covariance is the kernel $\kappa(x,x')$, and it depends on $f$

For each activation we draw 20 000 random nets of width $N=200$. The left panel shows their covariance $\mathrm{Cov}[\phi(x), \phi(x')]$ at fixed $x'=1$, which estimates $\kappa(x, 1)$. The other panels show five random networks (single draws, $N = 2000$) together with the band $\pm 2\sqrt{\kappa(x,x)}$, again estimated from the 20 000 nets.

The kernels are **not stationary**: they depend on $x$ and $x'$ separately, not only on $x-x'$ as the RBF kernel from the GP lecture does.

- **linear**: straight lines, the BLR prior.
- **tanh**: the variance saturates at large $|x|$, and the functions become flat.
- **ReLU**: the variance grows as $x^2$, and the functions are almost linear far out with kinks in between.

In [ ]:
xs = np.linspace(-3, 3, 61)
i1 = np.argmin(np.abs(xs - 1.0))          # index of x' = 1
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6))
for k, act_name in enumerate(["linear", "tanh", "relu"]):
    F = random_nets(xs, 200, act_name, M=20000, seed=10 + k)
    cov = (F - F.mean(0)).T @ (F[:, [i1]] - F[:, i1].mean()) / len(F)
    ax[0].plot(xs, cov.ravel(), color=f"C{k}", label=act_name)

    draws = random_nets(xs, 2000, act_name, M=5, seed=100 + k)
    band = 2*F.std(0)
    ax[k+1].fill_between(xs, -band, band, color=f"C{k}", alpha=0.15, label=r"$\pm2\sqrt{\kappa(x,x)}$")
    ax[k+1].plot(xs, draws.T, color=f"C{k}", lw=1)
    ax[k+1].set(title=f"random nets, $f$ = {act_name}", xlabel="$x$")
    ax[k+1].legend(loc="upper left", fontsize=9)
ax[0].set(xlabel="$x$", title=r"$\kappa(x, x'=1)$ from 20 000 random nets")
ax[0].legend()
plt.tight_layout()

**Deeper networks.** For $L$ hidden layers the same argument applies layer by layer, taking the widths to infinity one at a time. Each layer maps the kernel of the previous one:
$$
\kappa^{l+1}(x,x') = \sigma_b^2 + \sigma_w^2\,\mathbb{E}_{(u,u')\sim\mathcal{N}(0,\,K^l)}\bigl[f(u)\,f(u')\bigr],
$$
where $K^l$ is the $2\times 2$ matrix of $\kappa^l$ evaluated at $x, x'$ (Lee et al. 2018; Matthews et al. 2018). How this map behaves as the depth grows is the starting point of the book's chapter *ANNs in the large-width limit*, where finite-width corrections are organized in powers of depth/width.